# Step 1: Spark Environment Setup
## FedHome_Spark Project

**Student:** Md. Raihan Sobhan  
**Course:** Big Data Analytics  
**Date:** 2026-10-07

---

## Overview

This notebook sets up and verifies the Apache Spark environment for the FedHome_Spark project.

### Objectives
1. Initialize Spark session with optimal configuration
2. Test basic Spark operations
3. Verify Spark MLlib availability

---

## 1. Spark Session Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import BisectingKMeans
import pyspark

print(f"PySpark Version: {pyspark.__version__}")

In [ ]:
# Create Spark session optimized for MacBook Pro M5 (24GB RAM)
spark = SparkSession.builder \
    .appName("FedHome_Spark") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .config("spark.default.parallelism", "10") \
    .getOrCreate()

print("Spark Session Created Successfully!")
print(f"Spark UI: http://localhost:4040")
print(f"Available Cores: {spark.sparkContext.defaultParallelism}")

---

## 2. Basic Spark Operations Test

In [ ]:
from pyspark.sql import functions as F

# Create test DataFrame
data = [(i, f"user_{i}", i * 1.5) for i in range(10000)]
df = spark.createDataFrame(data, ["id", "name", "value"])

print(f"DataFrame Count: {df.count()}")
print(f"Schema:")
df.printSchema()

# Test aggregation
df.agg(
    F.avg("value").alias("avg_value"),
    F.max("value").alias("max_value"),
    F.min("value").alias("min_value")
).show()

---

## 3. Spark MLlib Test - BisectingKMeans Clustering

In [ ]:
import numpy as np
np.random.seed(42)

# Create sample feature data (simulating device-type profiles)
sample_data = []
for i in range(1000):
    features = np.random.randn(10).tolist()
    sample_data.append((i, features))

features_df = spark.createDataFrame(sample_data, ["id", "features_raw"])
features_df.show(5)

In [ ]:
# Convert to vector format for MLlib
assembler = VectorAssembler(inputCols=["features_raw"], outputCol="features")
features_vector_df = assembler.transform(features_df).select("id", "features")
features_vector_df.show(5, truncate=False)

In [ ]:
# Run BisectingKMeans clustering
bkmeans = BisectingKMeans(k=5, maxIter=10, seed=42)
model = bkmeans.fit(features_vector_df)

print(f"\n✅ Clustering Complete!")
print(f"Number of clusters: {model.k}")
print(f"Cluster centers computed: {len(model.clusterCenters())}")

# Show cluster assignments
predictions = model.transform(features_vector_df)
print("\nCluster distribution:")
predictions.groupBy("prediction").count().orderBy("prediction").show()

---

## 4. Summary

### Environment Verification Results

| Component | Status | Details |
|-----------|--------|---------|
| Java | ✅ | OpenJDK 17 |
| PySpark | ✅ | Version 4.1.2 |
| Spark Session | ✅ | 16GB driver memory |
| DataFrame Ops | ✅ | Working |
| MLlib Clustering | ✅ | BisectingKMeans tested |

### Next Steps
1. ✅ Spark environment verified
2. Next: `02_fedmse_spark_fullscale.ipynb` for FedMSE baseline run
3. Then: `03_fedhome_spark_clustering.ipynb` for FedHome clustering

In [ ]:
# Cleanup
spark.stop()
print("Spark session stopped.")